# NB_AGT_02 · Sincronizar ejemplos e instrucciones con el data agent

Lleva al **borrador** (*staging*) del Fabric data agent lo que el área de negocio validó:

1. Las preguntas `VALIDADA` con uso `EJEMPLO` de `agente.banco_preguntas` → *example queries* de cada fuente SQL o KQL.
2. Opcional: los bloques de `instrucciones_agente.md` → instrucciones del agente, y descripción e instrucciones de cada fuente.

**No publica.** La publicación la hace `NB_AGT_03` solo si la evaluación aprueba. Antes de tocar nada guarda un backup de los ejemplos actuales en `Files/agente/<AGENTE>/backup/<fecha>/`.

Con `DRY_RUN = True` (default) muestra el plan sin cambiar el agente.

SDK: [`fabric-data-agent-sdk`](https://learn.microsoft.com/en-us/fabric/data-science/fabric-data-agent-sdk), en preview. Probado contra el código de la versión `0.1.33a0`, no contra un agente real.

In [ ]:
%pip install -q fabric-data-agent-sdk==0.1.33a0

In [ ]:
AGENTE = "AG_VENTAS"              # nombre del data agent (también separa el banco)
WORKSPACE = None                   # None = el workspace del notebook
TABLA = "agente.banco_preguntas"
RUTA_INSTRUCCIONES = ""            # ej. /lakehouse/default/Files/agente/AG_VENTAS/instrucciones_agente.md ; vacío = no tocar instrucciones
BORRAR_NO_GESTIONADOS = False      # True: borra ejemplos del agente que no están en el banco
DRY_RUN = True

In [ ]:
# --- funciones (sin Fabric: las usa pruebas/prueba_local.py) ---
import re


def normalizar(texto: str) -> str:
    t = re.sub(r"[¿?¡!.,;:]", " ", str(texto).lower())
    return re.sub(r"\s+", " ", t).strip()


def plan_ejemplos(deseados: dict, actuales: list, borrar_no_gestionados: bool = False) -> list:
    """deseados: {pregunta: consulta} del banco. actuales: [{id, question, query}] del agente.
    Devuelve una lista de acciones: AGREGAR, ACTUALIZAR, BORRAR, SIN_CAMBIOS o NO_GESTIONADO."""
    por_pregunta = {normalizar(a["question"]): a for a in actuales}
    acciones, vistos = [], set()
    for pregunta, consulta in deseados.items():
        n = normalizar(pregunta)
        vistos.add(n)
        actual = por_pregunta.get(n)
        if actual is None:
            acciones.append({"accion": "AGREGAR", "id": None, "question": pregunta, "query": consulta})
        elif (actual.get("query") or "").strip() != consulta.strip():
            acciones.append({"accion": "ACTUALIZAR", "id": actual["id"], "question": pregunta, "query": consulta})
        else:
            acciones.append({"accion": "SIN_CAMBIOS", "id": actual["id"], "question": pregunta, "query": consulta})
    for n, a in por_pregunta.items():
        if n not in vistos:
            acciones.append({"accion": "BORRAR" if borrar_no_gestionados else "NO_GESTIONADO",
                             "id": a["id"], "question": a["question"], "query": a.get("query")})
    return acciones


def leer_bloques(markdown: str) -> dict:
    """Extrae los bloques <!-- BLOQUE:nombre --> ... <!-- /BLOQUE --> de instrucciones_agente.md."""
    patron = re.compile(r"<!--\s*BLOQUE:(.+?)\s*-->\s*(.*?)\s*<!--\s*/BLOQUE\s*-->", re.S)
    return {m.group(1).strip(): m.group(2).strip() for m in patron.finditer(markdown)}


def nombre_fuente(config: dict) -> str:
    """El nombre visible de una fuente en la respuesta del SDK (la clave cambió entre versiones)."""
    for clave in ("displayName", "display_name", "name"):
        if config.get(clave):
            return config[clave]
    return config.get("id", "?")


def filas_a_dicts(df) -> list:
    """Ejemplos del SDK (DataFrame) a [{id, question, query}], sin importar mayúsculas en las columnas."""
    if df is None or len(df) == 0:
        return []
    df = df.rename(columns=str.lower)
    return [{"id": r.get("id"), "question": r.get("question"), "query": r.get("query")}
            for r in df.to_dict("records")]

In [ ]:
# --- banco validado ---
banco = spark.sql(f"""
    SELECT pregunta_id, fuente, lenguaje, pregunta, consulta_referencia
    FROM {TABLA}
    WHERE agente = '{AGENTE}' AND uso = 'EJEMPLO' AND estado_efectivo = 'VALIDADA'
""").toPandas()
print(f"{len(banco)} ejemplos validados en {banco.fuente.nunique()} fuentes")

In [ ]:
# --- agente y backup ---
import json, os, datetime as dt
from fabric.dataagent.client import FabricDataAgentManagement

agent = FabricDataAgentManagement(AGENTE, WORKSPACE)
fuentes = {nombre_fuente(ds.get_configuration()): ds for ds in agent.list_datasources()}
print("Fuentes del agente:", list(fuentes))

sello = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
carpeta = f"/lakehouse/default/Files/agente/{AGENTE}/backup/{sello}"
os.makedirs(carpeta, exist_ok=True)
with open(f"{carpeta}/settings.json", "w", encoding="utf-8") as f:
    json.dump(agent.get_settings(), f, ensure_ascii=False, indent=2, default=str)
actuales = {}
for nombre, ds in fuentes.items():
    with open(f"{carpeta}/fuente_{re.sub(r'[^A-Za-z0-9_-]', '_', nombre)}.json", "w", encoding="utf-8") as f:
        json.dump(ds.get_configuration(), f, ensure_ascii=False, indent=2, default=str)
    try:
        actuales[nombre] = filas_a_dicts(ds.get_fewshots())
    except Exception as ex:  # los modelos semánticos no tienen example queries
        print(f"  {nombre}: sin ejemplos ({type(ex).__name__})")
        actuales[nombre] = None
    with open(f"{carpeta}/ejemplos_{re.sub(r'[^A-Za-z0-9_-]', '_', nombre)}.json", "w", encoding="utf-8") as f:
        json.dump(actuales[nombre], f, ensure_ascii=False, indent=2, default=str)
print("Backup en", carpeta)

In [ ]:
# --- plan ---
import pandas as pd
planes = {}
for fuente, g in banco.groupby("fuente"):
    if fuente not in fuentes:
        print(f"ATENCIÓN: la fuente '{fuente}' del banco no está en el agente. Se omite.")
        continue
    if actuales.get(fuente) is None:
        print(f"ATENCIÓN: '{fuente}' no admite example queries (¿modelo semántico?). Se omite.")
        continue
    deseados = dict(zip(g.pregunta, g.consulta_referencia))
    planes[fuente] = plan_ejemplos(deseados, actuales[fuente], BORRAR_NO_GESTIONADOS)

resumen = pd.DataFrame([{"fuente": f, **a} for f, acc in planes.items() for a in acc])
if len(resumen):
    display(resumen.groupby(["fuente", "accion"]).size().rename("ejemplos").reset_index())
    display(resumen[resumen.accion != "SIN_CAMBIOS"])

bloques = {}
if RUTA_INSTRUCCIONES:
    with open(RUTA_INSTRUCCIONES, encoding="utf-8") as f:
        bloques = leer_bloques(f.read())
    print("Bloques de instrucciones:", list(bloques))

In [ ]:
# --- aplicar en el borrador (cambio con impacto: solo con DRY_RUN = False) ---
if DRY_RUN:
    print("DRY_RUN = True: el agente no cambió.")
else:
    for fuente, acciones in planes.items():
        ds = fuentes[fuente]
        for a in acciones:
            if a["accion"] == "AGREGAR":
                ds.add_fewshot(question=a["question"], query=a["query"])
            elif a["accion"] == "ACTUALIZAR":
                ds.update_fewshot(a["id"], question=a["question"], query=a["query"])
            elif a["accion"] == "BORRAR":
                ds.delete_fewshot(a["id"])
    if "instrucciones_agente" in bloques:
        agent.update_settings(ai_instructions=bloques["instrucciones_agente"])
    for nombre, ds in fuentes.items():
        instr = bloques.get(f"fuente:{nombre}:instrucciones")
        desc = bloques.get(f"fuente:{nombre}:descripcion")
        if instr or desc:
            try:
                ds.update_configuration(instructions=instr, description=desc)
            except Exception as ex:  # los modelos semánticos no admiten instrucciones de fuente
                print(f"  {nombre}: no se actualizaron instrucciones ({ex})")
    print("Borrador actualizado. Siguiente paso: NB_AGT_03 con ETAPA = 'sandbox'.")
    print("Para volver atrás: restaurá los ejemplos desde", carpeta, "o descartá el borrador con agent.reset_staging()")